In [ ]:
"""
Full-scale pull: all 48 qualified 2026 World Cup teams, raw (unaggregated)
player club stats across 2023-2025.

Run in TWO stages, in order:
  STAGE A: resolve + verify all 48 team IDs (48 calls, cheap)
           -- STOP and manually check the printed output before Stage B
  STAGE B: pull raw player stats for every roster, all 3 seasons
           (~3,850 calls total -- only run after Stage A looks correct)

Saves incrementally (one team at a time) so an interruption partway
through doesn't lose everything already pulled.
"""

import requests
import pandas as pd
import time
import os

API_KEY = "masked"
BASE_URL = "https://v3.football.api-sports.io"
HEADERS = {"x-apisports-key": API_KEY}
DELAY_BETWEEN_CALLS = 0.3  # Pro plan: 300 req/min -- 0.3s gives comfortable margin

SEASONS = (2023, 2024, 2025)

QUALIFIED_TEAMS_2026 = [
    "Canada", "Brazil", "Paraguay", "Morocco", "Norway", "France", "Mexico",
    "England", "Belgium", "United States", "Spain", "Portugal", "Switzerland",
    "Egypt", "Argentina", "Colombia",
    "South Africa", "Japan", "Germany", "Netherlands", "Ivory Coast", "Sweden",
    "Ecuador", "DR Congo", "Senegal", "Bosnia and Herzegovina", "Austria",
    "Croatia", "Algeria", "Australia", "Cape Verde", "Ghana",
    "Czech Republic", "South Korea", "Qatar", "Scotland", "Haiti", "Turkey",
    "Curaçao", "Tunisia", "New Zealand", "Iran", "Saudi Arabia", "Uruguay",
    "Iraq", "Jordan", "Uzbekistan", "Panama",
]
assert len(QUALIFIED_TEAMS_2026) == 48


def api_get(endpoint: str, params: dict, max_retries: int = 5) -> dict:
    for attempt in range(max_retries):
        resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params)
        remaining_day = resp.headers.get("x-ratelimit-requests-remaining")
        if remaining_day is not None and int(remaining_day) <= 20:
            print(f"  WARNING: only {remaining_day} daily requests left.")
        if resp.status_code == 429:
            retry_after = int(resp.headers.get("Retry-After", 0))
            wait_time = retry_after if retry_after > 0 else (2 ** attempt) * 5
            print(f"  Rate limited. Waiting {wait_time}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(wait_time)
            continue
        resp.raise_for_status()
        return resp.json()
    raise Exception("Still rate-limited after max retries.")



# STAGE A: Resolve and verify all 48 team IDs before spending the big
# batch of player-stat calls. National teams in this API are typically
# tagged as their own "team" entity distinct from club teams of the same
# name -- verify by checking the country/national fields in the output.

def resolve_all_team_ids(nations=QUALIFIED_TEAMS_2026) -> pd.DataFrame:
    rows = []
    for nation in nations:
        data = api_get("teams", {"name": nation})["response"]
        if not data:
            rows.append({"nation": nation, "team_id": None, "matched_name": None,
                         "country": None, "national": None, "status": "NOT FOUND"})
            print(f"  [MISS] {nation}")
        else:
            # Prefer an entry explicitly flagged as a national team if present
            national_entries = [t for t in data if t["team"].get("national")]
            best = national_entries[0] if national_entries else data[0]
            rows.append({
                "nation": nation,
                "team_id": best["team"]["id"],
                "matched_name": best["team"]["name"],
                "country": best["team"]["country"],
                "national": best["team"].get("national"),
                "status": "OK" if national_entries else "CHECK -- no 'national' flag found",
            })
            flag = "OK" if national_entries else "CHECK"
            print(f"  [{flag}] {nation} -> id={best['team']['id']}  name={best['team']['name']}  national={best['team'].get('national')}")
        time.sleep(DELAY_BETWEEN_CALLS)
    return pd.DataFrame(rows)


def retry_missed_teams(missed_nations: list) -> pd.DataFrame:
    """
    Try alternate spellings for teams that didn't match on the first pass.
    API-Football's naming convention sometimes differs from FIFA's
    official names or common media names.
    """
    candidates = {
        "United States": ["USA", "United States of America"],
        "DR Congo": ["Congo DR", "DR Congo", "Congo"],
        "Bosnia and Herzegovina": ["Bosnia-Herzegovina", "Bosnia"],
        "Cape Verde": ["Cape Verde Islands", "Cabo Verde"],
        "Czech Republic": ["Czechia"],
        "Turkey": ["Türkiye", "Turkiye"],
    }

    rows = []
    for original_name in missed_nations:
        found = False
        for alt_name in candidates.get(original_name, []):
            data = api_get("teams", {"name": alt_name})["response"]
            if data:
                national_entries = [t for t in data if t["team"].get("national")]
                best = national_entries[0] if national_entries else data[0]
                rows.append({
                    "nation": original_name, "team_id": best["team"]["id"],
                    "matched_name": best["team"]["name"], "country": best["team"]["country"],
                    "national": best["team"].get("national"),
                    "status": f"OK (matched via alt spelling '{alt_name}')",
                })
                print(f"  [FOUND] {original_name} -> tried '{alt_name}' -> id={best['team']['id']}  name={best['team']['name']}")
                found = True
                break
            time.sleep(DELAY_BETWEEN_CALLS)
        if not found:
            rows.append({"nation": original_name, "team_id": None, "matched_name": None,
                         "country": None, "national": None, "status": "STILL NOT FOUND -- needs manual lookup"})
            print(f"  [STILL MISSING] {original_name} -- tried: {candidates.get(original_name, [])}")
    return pd.DataFrame(rows)



# STAGE A (REVISED): Pull the official 48-team list directly from the
# World Cup competition itself, rather than guessing name spellings for
# each nation individually. This guarantees IDs and names exactly as
# API-Football's own database defines them -- no manual retry needed.

def find_world_cup_league_id():
    """
    Find the league ID for the World Cup FINALS specifically (not
    qualifiers -- those are separate competitions). Prints all matches
    with type/country so you can visually confirm before picking one.
    """
    data = api_get("leagues", {"name": "World Cup"})["response"]
    print("Candidate competitions matching 'World Cup':")
    for entry in data:
        info = entry["league"]
        country = entry["country"]
        seasons = [s["year"] for s in entry["seasons"]]
        print(f"  id={info['id']:>6}  name={info['name']!r:30}  type={info['type']:12}  "
              f"country={country['name']:15}  seasons={seasons[-3:]}")
    return data  


def get_official_team_list(league_id: int, season: int) -> pd.DataFrame:
    """
    Pull the actual 48 teams registered to the World Cup competition/
    season, directly from the API -- exact IDs and names, no guessing.
    """
    data = api_get("teams", {"league": league_id, "season": season})["response"]
    rows = []
    for entry in data:
        team = entry["team"]
        rows.append({
            "team_id": team["id"],
            "matched_name": team["name"],
            "country": team["country"],
            "national": team.get("national"),
            "status": "OK (from official competition team list)",
        })
    df = pd.DataFrame(rows)
    print(f"\nPulled {len(df)} teams from league_id={league_id}, season={season}")
    if len(df) != 48:
        print(f"WARNING: expected 48 teams, got {len(df)} -- double check league_id/season "
              f"(e.g. this may have pulled qualifiers instead of the finals).")
    return df



# STAGE B: Raw, unaggregated player stats pull for every team/player/season.
# No aggregation -- one row per (player, season, competition) exactly as
# returned by the API. Saves incrementally per team.

def get_squad(team_id: int) -> pd.DataFrame:
    data = api_get("players/squads", {"team": team_id})["response"]
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data[0]["players"])
    df["team_id"] = team_id
    df["team_name"] = data[0]["team"]["name"]
    return df


def get_player_club_stats_raw(player_id: int, season: int) -> pd.DataFrame:
    data = api_get("players", {"id": player_id, "season": season})
    if data["errors"] or not data["response"]:
        return pd.DataFrame()

    rows = []
    profile = data["response"][0]["player"]
    for stat_block in data["response"][0]["statistics"]:
        games = stat_block.get("games", {}) or {}
        goals = stat_block.get("goals", {}) or {}
        passes = stat_block.get("passes", {}) or {}
        shots = stat_block.get("shots", {}) or {}
        tackles = stat_block.get("tackles", {}) or {}
        duels = stat_block.get("duels", {}) or {}
        rows.append({
            "player_id": profile["id"], "name": profile["name"], "age": profile["age"],
            "nationality": profile["nationality"], "season": season,
            "club": stat_block.get("team", {}).get("name"),
            "competition": stat_block.get("league", {}).get("name"),
            "country": stat_block.get("league", {}).get("country"),
            "appearances": games.get("appearences"), "minutes": games.get("minutes"),
            "position": games.get("position"), "rating": games.get("rating"),
            "goals": goals.get("total"), "assists": goals.get("assists"),
            "passes_total": passes.get("total"), "passes_key": passes.get("key"),
            "pass_accuracy_pct": passes.get("accuracy"),
            "shots_total": shots.get("total"), "shots_on_target": shots.get("on"),
            "tackles_total": tackles.get("total"),
            "duels_total": duels.get("total"), "duels_won": duels.get("won"),
        })
    return pd.DataFrame(rows)


def pull_all_teams_raw(team_ids_df: pd.DataFrame, seasons=SEASONS, out_dir="raw_pull"):
    os.makedirs(out_dir, exist_ok=True)
    master_path = os.path.join(out_dir, "all_players_raw_stats.csv")
    first_write = not os.path.exists(master_path)

    valid_teams = team_ids_df[team_ids_df["team_id"].notna()]
    for _, row in valid_teams.iterrows():
        nation, team_id = row["nation"], int(row["team_id"])
        print(f"\n--- {nation} (team_id={team_id}) ---")

        squad = get_squad(team_id)
        squad.to_csv(os.path.join(out_dir, f"squad_{nation.replace(' ', '_')}.csv"), index=False)
        time.sleep(DELAY_BETWEEN_CALLS)

        team_rows = []
        for _, player in squad.iterrows():
            for season in seasons:
                stats = get_player_club_stats_raw(player["id"], season)
                if not stats.empty:
                    stats["roster_nation"] = nation
                    team_rows.append(stats)
                time.sleep(DELAY_BETWEEN_CALLS)

        if team_rows:
            team_df = pd.concat(team_rows, ignore_index=True)
            team_df.to_csv(
                master_path, mode="a", header=first_write, index=False
            )
            first_write = False
            print(f"  Saved {len(team_df)} rows for {nation} -> appended to {master_path}")


if __name__ == "__main__":
    print("=== STAGE A-1: Finding the World Cup FINALS league ID ===")
    print("(review output below -- pick the id with type='Cup' and the right season)")
    find_world_cup_league_id()


    WORLD_CUP_LEAGUE_ID = 1  

    if WORLD_CUP_LEAGUE_ID is None:
        print("\n>>> STOP HERE. Set WORLD_CUP_LEAGUE_ID above based on the output, then re-run. <<<")
    else:
        print(f"\n=== STAGE A-2: Pulling official 48-team list (league_id={WORLD_CUP_LEAGUE_ID}) ===")
        team_ids_df = get_official_team_list(WORLD_CUP_LEAGUE_ID, season=2026)
        team_ids_df["nation"] = team_ids_df["matched_name"]  
        team_ids_df.to_csv("wc2026_team_ids.csv", index=False)
        print(f"Saved -> wc2026_team_ids.csv")
        print(team_ids_df[["nation", "team_id", "country", "national"]])

        print("\n>>> STOP HERE. Review wc2026_team_ids.csv manually before running Stage B. <<<")

        print("\n=== STAGE B: Pulling raw player stats for all 48 teams (2023-2025) ===")
        pull_all_teams_raw(team_ids_df, seasons=SEASONS)

=== STAGE A-1: Finding the World Cup FINALS league ID ===
(review output below -- pick the id with type='Cup' and the right season)
Candidate competitions matching 'World Cup':
  id=     1  name='World Cup'                     type=Cup           country=World            seasons=[2018, 2022, 2026]

=== STAGE A-2: Pulling official 48-team list (league_id=1) ===

Pulled 48 teams from league_id=1, season=2026
Saved -> wc2026_team_ids.csv
                  nation  team_id             country  national
0                Belgium        1             Belgium      True
1                 France        2              France      True
2                Croatia        3             Croatia      True
3                 Sweden        5              Sweden      True
4                 Brazil        6              Brazil      True
5                Uruguay        7             Uruguay      True
6               Colombia        8            Colombia      True
7                  Spain        9               Sp

: 